In [ ]:
!pip install transformers torch pandas scikit-learn

In [ ]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments

In [ ]:
import pandas as pd

df = pd.read_csv("/aa_dataset-tickets-en-only.csv")

In [ ]:
cols = [
    'subject','body','answer','queue','priority','version',
    'tag_1','tag_2','tag_3','tag_4','tag_5','tag_6','tag_7','tag_8','type'
]

df = df[cols]

In [ ]:
df.fillna("", inplace=True)

In [ ]:
df["text"] = (
    "subject: " + df["subject"].astype(str) +
    " body: " + df["body"].astype(str) +
    " answer: " + df["answer"].astype(str) +
    " queue: " + df["queue"].astype(str) +
    " priority: " + df["priority"].astype(str) +
    " version: " + df["version"].astype(str) +
    " tags: " +
    df["tag_1"].astype(str) + " " +
    df["tag_2"].astype(str) + " " +
    df["tag_3"].astype(str) + " " +
    df["tag_4"].astype(str) + " " +
    df["tag_5"].astype(str) + " " +
    df["tag_6"].astype(str) + " " +
    df["tag_7"].astype(str) + " " +
    df["tag_8"].astype(str)
)

In [ ]:
df[["text","type"]].head()

,text,type
0,subject: Account Disruption body: Dear Custome...,Incident
1,subject: Query About Smart Home System Integra...,Request
2,subject: Inquiry Regarding Invoice Details bod...,Request
3,subject: Question About Marketing Agency Softw...,Problem
4,subject: Feature Query body: Dear Customer Sup...,Request


In [ ]:
df.isnull().sum()

,0
subject,0
body,0
answer,0
queue,0
priority,0
version,0
tag_1,0
tag_2,0
tag_3,0
tag_4,0


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["type"])

print("Classes:", le.classes_)

Classes: ['Change' 'Incident' 'Problem' 'Request']


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"],
    df["label"],
    test_size=0.2,
    random_state=42
)

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
import torch

class TicketDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = TicketDataset(train_encodings, train_labels)
val_dataset = TicketDataset(val_encodings, val_labels)

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(df["label"].unique())
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.390051,0.391825
2,0.317729,0.333591
3,0.226949,0.413802


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4902, training_loss=0.3353124950818265, metrics={'train_runtime': 2166.0921, 'train_samples_per_second': 18.102, 'train_steps_per_second': 2.263, 'total_flos': 5158384868782080.0, 'train_loss': 0.3353124950818265, 'epoch': 3.0})

In [ ]:
text = "Unable to login into account after update"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model(**inputs)

pred = torch.argmax(outputs.logits)

print("Predicted type:", le.inverse_transform([pred.item()])[0])

Predicted type: Incident


accuracy checking

In [2]:
from sklearn.metrics import accuracy_score
import numpy as np

In [3]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [5]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

NameError: name 'Trainer' is not defined

In [1]:
trainer.train()

NameError: name 'trainer' is not defined

In [ ]:
results = trainer.evaluate()
print(results)